# 06 — Root finding and differentiable interpolation

**Objectives.** Solve parameterized implicit equations and embed lookup tables in differentiable graphs. For $g(z,p)=0$, the implicit sensitivity is $dz/dp=-(\partial g/\partial z)^{-1}(\partial g/\partial p)$ when the first Jacobian is nonsingular.


In [ ]:
import casadi as ca
import numpy as np
z = ca.MX.sym('z'); p = ca.MX.sym('p')
g = ca.Function('g', [z, p], [z**3 + z - p])
root = ca.rootfinder('root', 'newton', g)
solution = root(0.0, 2.0)
assert abs(float(solution**3 + solution - 2)) < 1e-10
root_of_p = root(0.0, p)
sensitivity = ca.Function('dz_dp', [p], [ca.jacobian(root_of_p, p)])
print('z(2)=', solution, 'dz/dp=', sensitivity(2.0))


In [ ]:
grid = np.linspace(0, 4, 9)
values = np.sqrt(grid + 1)
linear = ca.interpolant('linear_table', 'linear', [grid], values)
spline = ca.interpolant('spline_table', 'bspline', [grid], values)
query = ca.MX.sym('query')
slope = ca.Function('slope', [query], [ca.jacobian(spline(query), query)])
print(float(linear(1.25)), float(spline(1.25)), float(slope(1.25)))


**Failure modes:** singular implicit Jacobians, poor initial guesses, extrapolation, and spline overshoot. `rootfinder.stats()` reports termination information. **Recap:** root finders and interpolants are Functions, so their outputs can participate in larger derivative graphs.
